# Introduction

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **pooled data** and compute *stationary GEV analysis* for 
**annually grouped data**.<br>

**Options**
- **Option1**<br>
    pool all data per location for global stationary and non-stationary analysis<br><br>
- **Option2**<br>
    per year grouped data per location for stationary analysis<br>

---
**Workflow Summary (per location)**
1. Extract annual maxima → fit stationary & non-stationary GEV per model
2. Compute return levels & CIs per year per model
3. Compute exceedance probability & CI for thresholds of interest
4. Aggregate multi-model ensemble: mean + total spread
5. Visualization
   1. Return levels vs year (shaded CI)
   2. Probability amplification vs year (shaded CI)
   3. Multi-model ensemble bar plots for future RL
   4. Maps (mean & spread)
6. Tables:
   1. RL per T per year ± CI
   2. Probability change of historical RL


**CHECK** STD Calculation! it is basically 0 and doesnt really make sense 

# Import Packages

In [ ]:
from typing import Optional, Tuple

import sys
import random
import time


import xarray as xr
from pandas import concat, Series
from scipy import optimize, stats
from scipy.optimize import approx_fprime, minimize
from scipy.stats import norm, genextreme, chi2
from numpy import (
    ndarray, full_like, inf, exp, zeros, std, array, mean, diag, min, max, sqrt, log, 
    arange, ones, sum, linalg, random, isfinite, finfo, all, isfinite, 
    full_like, asarray, repeat
)

In [1]:
from typing import Optional

import os
import pickle
from datetime import datetime
from glob import glob
from pathlib import Path
import warnings
import random

from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
from collections import OrderedDict

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import cmcrameri.cm as cmc

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

from scipy.stats import genextreme
import statsmodels.api as sm
from pandas import DataFrame, concat
from numpy import arange, percentile, mean, std, ndarray, isfinite, all, sqrt, diag

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/.venv/lib/python3.12/site-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# Settings

In [2]:
path_input = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/'

hindcast_start = 1960
hindcast_end = 2026

return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']
ls_t_eval = 2026, 2050

# adding a little randomness to enhance trust the model works as robust as possible cross locations..
start_location = random.choice(arange(0, 9579))
end_location = start_location+10
print(f'analyse a subsample of location {start_location}–{end_location}')


colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

_LOCATION_LABELS = None
export_report=True
display_results = False

analyse a subsample of location 5244–5254


# Import Data

In [3]:
ls_files = [file for file in glob(path_input + '*.nc')]

print('Importing Data from ...')
print("\n".join(ls_files))

dic_data_per_model = dbf.import_all_models(ls_files)

Importing Data from ...
../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc
../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc
../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc
../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc
../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc
../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc
../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc
../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc


# Prepare Data

### Pooling, BiasCorrection, ValidityCheck

In [4]:
print('Pooling and Preparing Data...')
dic_data_per_model, combined, notes_overview = dbf.prepare_combined_data(ls_files, dic_data_per_model)
print('... done.')

Pooling and Preparing Data...
... done.


### Rearrangement to per location

In [5]:
print('Rearranging Data – sorting per location...')    
dic_data_per_location = dbf.extract_location_data(combined, hindcast_start, hindcast_end)

Rearranging Data – sorting per location...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    6.8s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    7.0s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    7.1s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:    7.1s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    7.2s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19885573509666093s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    7.3s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:    7.7s
[Parallel(n_jobs=-1)]: Done 108 tasks      | elapsed:    7.9s
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Done 202 tasks      | elapsed:    8.7s
[Parallel(n_jobs=-1)]: Done 236 tasks      | elapsed:    9.0s
[Parallel(n_jobs=-1)]:

## Select Subset

In [6]:
if start_location is not None or end_location is not None: 
    dic_data_per_location = ut.select_allowed_locations(
        dic_data_per_location=dic_data_per_location, 
        start_loc=start_location, end_loc=end_location
        )
    print(f'Processing locations {start_location} to {end_location} ({len(dic_data_per_location)} total)')

else:
    print(f'Processing all {len(dic_data_per_location)} locations')

print('Getting closest point available as location label for orientation. \nNote this is not the exact location...')
location_labels = dbf.precompute_location_labels(dic_data_per_location)
location_labels

Processing locations 5244 to 5254 (11 total)
Getting closest point available as location label for orientation. 
Note this is not the exact location...
Loading formatted geocoded file...


{(4.528193, 36.892272): 'Azazga Tizi Ouzou DZ',
 (4.535245, 52.396322): 'Zandvoort North Holland NL',
 (4.551236, 52.420815): 'Zandvoort North Holland NL',
 (4.557514, 52.457381): 'Bloemendaal North Holland NL',
 (4.557936, 52.444341): 'Zandvoort North Holland NL',
 (4.561868, 52.473196): 'Velsen-Zuid North Holland NL',
 (4.579802, 52.489294): 'Velsen-Zuid North Holland NL',
 (4.585525, 43.56597): "Arles Provence-Alpes-Cote d'Azur FR",
 (4.586792, 52.502987): 'Beverwijk North Holland NL',
 (4.586969, 36.886186): 'Azazga Tizi Ouzou DZ',
 (4.593454, 52.465846): 'Velsen-Zuid North Holland NL'}

# (Non-)Stationary GEV Analysis with Pooled Data

When fitting a GEV to annual maxima, there are multiple sources of uncertainty in GEV analysis
- Parameter uncertainty (estimation uncertainty)
- Natural variability / return level uncertainty

**Parameter uncertainty**<br>
The GEV has parameters $(μ, σ, ξ)$ <br>
Each of these parameters is estimated from finite data, so each has an associated uncertainty.
This “uncertainty in the location parameter" can be captured by:
- Parametric sampling: sample $μ ~ Normal(μ, SE_μ)$
- Bootstrap: refit GEV on resampled data
- Delta Method (because the others are too instable)

**Natural variability / return level uncertainty** <br>
Even if parameters were known exactly, extreme values themselves are random.
The return level $z_T$ is defined as a high quantile of the GEV:<br>
$z_T = μ + σ/ξ · [(-ln(1-1/T))^ξ - 1]$
<br>

When sampling from fitted GEV distribution, we can get a confidence interval for $z_T$ given fixed parameters.
This is the *return level uncertainty* captured by `gev_return_levels_ci`.

**Key Differences**
|Concept| How it's captured| Effect|
|---|---|---|
|μ (location) uncertainty	|Parametric bootstrap or μ-sampling from estimated SE	|Adds spread to the estimated parameter itself|
|Return level uncertainty	|Sampling from GEV with fixed parameters	|Adds spread due to natural variability of extremes|
|Combined uncertainty	|Sample μ from its distribution, then compute z_T from each μ	|Gives realistic CIs for return levels, including both sources|


In practice, best approach:
- Sample μ (location) from its uncertainty distribution
- For each μ, compute the return level $z_T$ using the quantile formula
- Compute percentile (median, 95% CI) → includes both parameter uncertainty and natural variability

**EXPECTATION TOWARDS VALUES**<br>
In a stationary Generalized Extreme Value distribution:
- μ(mu) = constant → location parameter
- σ(sigma) = constant → scale parameter
- ξ(xi) = constant → shape parameter<br>

So the distribution does not depend on time.
Therefore, $z_T = f(μ,σ,ξ)$ depends only on T, not on time.

---
**NOTE**
- uncertainty is computed using delta method
- for return levels, the uncertainty in sigma, xi are ignored
- weighted nonstationary STDs >>> tried Fisher, Bootstrap but all unreasonable values... go back to delta method

## Compute for 1 Sample

In [7]:
def set_location_labels(labels):
    global _LOCATION_LABELS
    _LOCATION_LABELS = labels

In [8]:
## UPDATE - testing for 1 location | later to be parallelized
# include uncertainty in stationary GEV parameters
uncertainty = 'delta'
B = 300 # bootstrap number - downscaling to 50?! (from 300)
seed = None

# ----------------------------------------------------------------------------------------------------------
results = dict()
loc_id = list(dic_data_per_location.keys())[0]
df_prepared = dic_data_per_location[loc_id]

lon_loc = df_prepared.lon.unique()[0]
lat_loc = df_prepared.lat.unique()[0]

set_location_labels(location_labels)
location_info = _LOCATION_LABELS.get((round(lon_loc,6), round(lat_loc,6)), "unknown location")
print(f'GEV analysis for location {loc_id} · {location_info}')

# ----------------------------------------------------------------------------------------------------------
ls_notes = []
annual_max = gev.extract_annual_maxima_at_location(df_prepared, lon=lon_loc, lat=lat_loc)

if len(annual_max) < 10:
    print('WARNING - not enough data (<10) for location {loc_id} (lon|lat · {lon_loc}|{lat_loc})')

years = annual_max['year'].values
data = annual_max['annual_max'].values

if uncertainty is None:
    uncertainty = "delta"

ls_notes = []
n = len(data)

# ----------------------------------------------------------------------------------------------------------
print("\tConducting (non-)stationary GEV with pooled data...")
pooled_gev, ls_notes = gev.fit_pooled_gev_with_uncertainty(
    loc_id=loc_id, years=years, data=data, uncertainty=uncertainty, trend_params='location', print_msg=True
    )
print(f"\t → (Non-)Stationary GEV done (success {pooled_gev != None})")

print("\tCompare Models...")
comparison = gev.compare_stationary_nonstationary(pooled_gev['stationary'], pooled_gev['nonstationary'], annual_max)

print("\tCompute Return Levels...")
#all_return_levels = dict()
#for T in return_periods:
#    rl = gev.compute_return_levels_for_year(pooled_gev['stationary'], pooled_gev['nonstationary'], T=T, t_eval=ls_t_eval)
#    all_return_levels[T] = rl
#df_all_return_levels = gev.convert_return_level_format(return_periods, ls_t_eval, all_return_levels)
                            
results[loc_id] = dict({
    'location_info': location_info,
    'LatLon': (lat_loc, lon_loc),
    'data':annual_max, 
    'stationary': pooled_gev['stationary'],
    'nonstationary': pooled_gev['nonstationary'],
    'model_comparison': comparison,
#    'return_levels': df_all_return_levels
    })

GEV analysis for location 5244 · Azazga Tizi Ouzou DZ
	Conducting (non-)stationary GEV with pooled data...
		Compute stationary GEV incl uncertainty using delta
[[nan nan nan nan]
 [nan nan nan nan]
 [nan nan nan nan]
 [nan nan nan nan]]
[[nan nan nan nan]
 [nan nan nan nan]
 [nan nan nan nan]
 [nan nan nan nan]]
	 → (Non-)Stationary GEV done (success True)
	Compare Models...
	Compute Return Levels...


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/.venv/lib/python3.12/site-packages/numdifftools/limits.py:199: UserWarning: All-NaN slice encountered
  idx = _Limit._get_arg_min(errors)
/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [65]:
from numpy import (all, any, arange, array, asarray, diag, exp, finfo, float64,
                   full_like, generic, inf, isfinite, isnan, linalg, linspace,
                   log, max, mean, min, nan, ndarray, ones, ones_like,
                   percentile, random, repeat, sqrt, std, sum, vstack, zeros,
                   zeros_like)
from numpy.linalg import inv
import numdifftools as nd
from scipy.optimize import approx_fprime, minimize
from statsmodels.tools.numdiff import approx_hess
from scipy import stats, linalg



def compute_return_levels_for_year(stationary, nonstationary, T, t_eval):
    t_eval = asarray(t_eval)
    
    #  stationary
    mu = stationary['location']
    sd_mu = stationary['location_std']
    scale = stationary['scale']
    shape = stationary['shape']
    
    factor = (-log(1 - 1/T))**(-shape) - 1
    z_T = mu + scale/shape * factor
    z_lower = z_T - 1.96 * sd_mu
    z_upper = z_T + 1.96 * sd_mu

    # non-stationary
    params_hat = nonstationary['params_hat']
    years_mean = nonstationary['years_mean']
    years_std = nonstationary['years_std']
    
    mu0, mu1, sigma, xi = params_hat
    t_scaled_eval = (t_eval - years_mean) / years_std
    
    #mu_t = nonstationary['mu0_samples'].mean() + nonstationary['mu1_samples'].mean()*t_scaled_eval
    mu_t = mu0 + mu1*t_scaled_eval
    #mu_samples_eval = nonstationary['mu0_samples'][:, None] + nonstationary['mu1_samples'][:, None]*t_scaled_eval
    
    if 'delta_sd_mu' in nonstationary:
        sd_mu_t = array([nonstationary['delta_sd_mu'](year) for year in t_eval])
    else:
        sd_mu_t = zeros_like(mu_t)  # fallback if Delta-method not available

    if abs(xi) < 1e-10:
        factor_ns = -log(-log(1 - 1/T))
        z_T_ns = mu_t + sigma * factor_ns
    else:
        factor_ns = (-log(1 - 1/T))**(-xi) - 1
        z_T_ns = mu_t + sigma/xi * factor_ns

    z_lower_ns = z_T_ns - 1.96 * sd_mu_t
    z_upper_ns = z_T_ns + 1.96 * sd_mu_t
    
    return dict({
        'stationary': {'z_T': z_T, 'lower': z_lower, 'upper': z_upper},
        'nonstationary': {'z_T': z_T_ns, 'lower': z_lower_ns, 'upper': z_upper_ns},
            })


def compute_cov_matrix(gev_params: dict, data: ndarray, years: ndarray = None) -> ndarray:
    """
    Compute the covariance matrix of fitted GEV parameters using a numerically stable Hessian.
    
    Parameters
    ----------
    gev_params : dict
        Fitted GEV parameters. Should contain:
        - 'shape', 'location', 'scale' for stationary
        - or 'mu0', 'mu1', 'sigma', 'xi' for non-stationary location trend
    data : np.ndarray
        Observed annual maxima used for fitting
    years : np.ndarray, optional
        Years array (required if non-stationary model with trend)
    
    Returns
    -------
    cov_matrix : np.ndarray
        Covariance matrix of parameters, or None if Hessian is singular.
    """
    messages = ''
    if 'trend_in' in gev_params and gev_params['trend_in'] == 'location':
        # Non-stationary location μ(t) = μ0 + μ1 * t
        t = (years - gev_params['years_mean']) / gev_params['years_std']

        def neg_loglik(theta):
            mu0, mu1, log_sigma, xi = theta
            sigma = exp(log_sigma)
            mu_t = mu0 + mu1 * t
            c = -xi  # SciPy convention
            z = 1 + xi * (data - mu_t)/sigma

            # Penalize invalid parameters (support violation or sigma <= 0)
            if any(z <= 1e-10) or sigma <= 0:
                return 1e10

            ll = stats.genextreme.logpdf(data, c, loc=mu_t, scale=sigma)
            if not all(isfinite(ll)):
                return 1e10

            return -sum(ll)

        theta_hat = array([
            gev_params.get('mu0', gev_params['params_hat'][0]),
            gev_params.get('mu1', gev_params['params_hat'][1]),
            log(gev_params.get('sigma', gev_params['params_hat'][2])),
            gev_params.get('xi', gev_params['params_hat'][3])
        ])

    else:
        # Stationary GEV: μ, σ, ξ
        def neg_loglik(theta):
            xi, mu, log_sigma = theta
            sigma = exp(log_sigma)
            z = 1 + xi * (data - mu)/sigma
            if any(z <= 1e-10) or sigma <= 0:
                return 1e10
            ll = stats.genextreme.logpdf(data, -xi, loc=mu, scale=sigma)
            if not all(isfinite(ll)):
                return 1e10
            return -sum(ll)

        theta_hat = array([
            gev_params['shape'],
            gev_params['location'],
            log(gev_params['scale'])
        ])

    try:
        if 'trend_in' in gev_params and gev_params['trend_in'] == 'location':
            mu0, mu1, log_sigma, xi = theta_hat
            sigma = exp(log_sigma)
            mu_t = mu0 + mu1 * t
            z = 1 + xi * (data - mu_t)/sigma
        else:
            xi, mu, log_sigma = theta_hat
            sigma = exp(log_sigma)
            z = 1 + xi * (data - mu)/sigma
        messages += f'\t\t\tMLE min(z):{min(z)}'
    except Exception:
        pass


    H = approx_hess(theta_hat, neg_loglik)
    eigvals = linalg.eigvals(H)
    messages += f'\n\t\t\tHessian eigenvalues: {eigvals}'

    try:
        cov_matrix = linalg.inv(H)
    except linalg.LinAlgError:
        print("Warning: Hessian not invertible; returning None")
        messages += "\n\t\t\tWarning: Hessian not invertible; returning None"
        return None
    print(messages)
    return cov_matrix, messages

In [77]:
## UPDATE - testing for 1 location | later to be parallelized
# include uncertainty in stationary GEV parameters
uncertainty = 'delta'
B = 300 # bootstrap number - downscaling to 50?! (from 300)
seed = None
print_msg = True
trend_params = 'location'


# ----------------------------------------------------------------------------------------------------------
results = dict()
loc_id = list(dic_data_per_location.keys())[0]
df_prepared = dic_data_per_location[loc_id]

lon_loc = df_prepared.lon.unique()[0]
lat_loc = df_prepared.lat.unique()[0]

set_location_labels(location_labels)
location_info = _LOCATION_LABELS.get((round(lon_loc,6), round(lat_loc,6)), "unknown location")
print(f'GEV analysis for location {loc_id} · {location_info}')

# ----------------------------------------------------------------------------------------------------------
ls_notes = []
annual_max = gev.extract_annual_maxima_at_location(df_prepared, lon=lon_loc, lat=lat_loc)

if len(annual_max) < 10:
    print('WARNING - not enough data (<10) for location {loc_id} (lon|lat · {lon_loc}|{lat_loc})')

years = annual_max['year'].values
data = annual_max['annual_max'].values

if uncertainty is None:
    uncertainty = "delta"

# ----------------------------------------------------------------------------------------------------------
print("\tConducting (non-)stationary GEV with pooled data...")
#pooled_gev, ls_notes = gev.fit_pooled_gev_with_uncertainty(
#    loc_id=loc_id, years=years, data=data, uncertainty=uncertainty, trend_params='location', print_msg=True
#    )
# ---- STEP 1: Get stationary ----
if print_msg:
    print(f'\t\tCompute stationary GEV incl uncertainty using {uncertainty}')
stationary, stat_notes = gev.fit_stationary_gev_incl_uncertainty(data)
ls_notes.extend(stat_notes)

if stationary is None:
    message = f"Failed to compute stationary GEV for location {loc_id}, skipping..."
    if print_msg:
        print(message)
    ls_notes.append(message)


# ---- STEP 2: Prepare for non-stationary GEV ----
if print_msg:
    print(f'\t\tCompute non-stationary GEV incl uncertainty using {uncertainty}')
    
if n < 20:
    message = f"Warning: Non-stationary GEV needs ≥20 observations but has {n}, "
    message += f"skipping non-stationary GEV for location {loc_id}..."
    if print_msg:
        print(message)
    ls_notes.append(message)

t = (years - years.mean()) / years.std()
x0 = [stationary['location'], 0.0, stationary['scale'], stationary['shape']]


# ---- STEP 3: Define negative log-likelihood for non-stationary case ----
def neg_loglik(params):
    if trend_params == 'location':
        mu0, mu1, sigma, xi = params
        mu_t = mu0 + mu1 * t
        sigma_t = full_like(t, sigma)
    elif trend_params == 'scale':
        mu, sigma0, sigma1, xi = params
        mu_t = full_like(t, mu)
        sigma_t = sigma0 + sigma1 * t
    elif trend_params == 'both':
        mu0, mu1, sigma0, sigma1, xi = params
        mu_t = mu0 + mu1 * t
        sigma_t = sigma0 + sigma1 * t
    else:
        raise ValueError("trend_params must be 'location', 'scale', or 'both'")
    
    if any(sigma_t <= 0):
        return inf
    
    z = (data - mu_t) / sigma_t
    if abs(xi) < 1e-10:  # Gumbel case
        ll = -sum(log(sigma_t)) - sum(z) - sum(exp(-z))
    else:
        term = 1 + xi * z
        if any(term <= 0):
            return inf
        ll = -sum(log(sigma_t)) - sum((1 + 1/xi) * log(term)) - sum(term**(-1/xi))
    
    return -ll
    
# ---- STEP 4: Fit MLE for non-stationary case ----
try:
    result = minimize(neg_loglik, x0, method='Nelder-Mead')
    params_hat = result.x
except Exception as e:
    message = f"Non-stationary MLE fitting failed: {e}"
    if print_msg:
        print(message)
    ls_notes.append(message)

print(f'\t\t\tnonstationary GEV - MLE {params_hat}')
nonstationary = {
    'params_hat': params_hat, 'trend_params': trend_params, 'n_obs': n, 
    'years_mean': years.mean(), 'years_std': years.std(), 'trend_in': trend_params
    }

# ---- STEP 5: Uncertainty Calculation for non-stationary case DEFAULT Fisher ----
if uncertainty == "delta":
    message = "\t\t\tcomputing delta method for non-stationary GEV..."
    ls_notes.append(message)
    if print_msg:
        print(message)
    try:
        cov_ns, hessian_eigenvalues = compute_cov_matrix(gev_params=nonstationary, data=data, years=years)
        nonstationary['Hessian eigenvalues'] = hessian_eigenvalues
    except:
        cov_ns = None
        ls_notes.append("Hessian inversion failed; Delta-method SD not computed.")

    if cov_ns is not None:
        nonstationary['cov_mu'] = cov_ns
    else:
        ls_notes.append(f"Delta Method failed, falling back to Fisher Information")
        uncertainty = "fisher"

print(f"\t → (Non-)Stationary GEV done (success {pooled_gev != None})")

print("\tCompare Models...")
comparison = compare_stationary_nonstationary(pooled_gev['stationary'], pooled_gev['nonstationary'], annual_max)

#print("\tCompute Return Levels...")
#all_return_levels = dict()
#for T in return_periods:
#    rl = gev.compute_return_levels_for_year(pooled_gev['stationary'], pooled_gev['nonstationary'], T=T, t_eval=ls_t_eval)
#    all_return_levels[T] = rl
#df_all_return_levels = gev.convert_return_level_format(return_periods, ls_t_eval, all_return_levels)
                            
results[loc_id] = dict({
    'location_info': location_info,
    'LatLon': (lat_loc, lon_loc),
    'data':annual_max, 
#    'stationary': pooled_gev['stationary'],
#    'nonstationary': pooled_gev['nonstationary'],
#    'model_comparison': comparison,
#    'return_levels': df_all_return_levels
    })

GEV analysis for location 5244 · Azazga Tizi Ouzou DZ
	Conducting (non-)stationary GEV with pooled data...
		Compute stationary GEV incl uncertainty using delta
		Compute non-stationary GEV incl uncertainty using delta
			nonstationary GEV - MLE [ 0.19433452 -0.00155958  0.04421027 -0.23651888]
			computing delta method for non-stationary GEV...
			MLE min(z):0.09671925001117343
			Hessian eigenvalues: [3322223.20848269+0.j 3279896.60391615+0.j   38494.06310109+0.j
    9932.10359659+0.j]
	 → (Non-)Stationary GEV done (success True)
	Compare Models...
123 6591
stationary 0.1943082930278949 0.04421614715082768 -0.23594547835147084 0       1.286005
1       0.978969
2       0.155165
3      -0.120170
4       0.823452
          ...   
6586    0.212654
6587    0.564648
6588   -0.598732
6589   -0.075019
6590    0.565912
Name: annual_max, Length: 6591, dtype: float64
non-stationary 0.19433451621010137 -0.001559577382467759 0.04421027236085026 -0.23651887838848482 0       1.215883
1       0.9088

In [78]:
comparison

{'stationary': {'LL': np.float64(11068.287076635448),
  'k': 3,
  'AIC': np.float64(-22130.574153270896),
  'BIC': np.float64(-22110.193772187737)},
 'nonstationary': {'LL': np.float64(11072.236105542936),
  'k': 4,
  'AIC': np.float64(-22136.47221108587),
  'BIC': np.float64(-22109.29836964166)},
 'LRT': {'delta_LL': np.float64(7.898057814974891),
  'df': 1,
  'p_value': np.float64(0.0049487905552759726),
  'interpretation': '→ non-stationary model is significantly better → μ(t) trend matters'}}

In [ ]:
all_return_levels = dict()
for T in return_periods:
    rl = compute_return_levels_for_year(pooled_gev['stationary'], pooled_gev['nonstationary'], T=T, t_eval=ls_t_eval)
    all_return_levels[T] = rl
df_all_return_levels = gev.convert_return_level_format(return_periods, ls_t_eval, all_return_levels)


,,,z_T,lower,upper
t_eval,return_period,model,,,
2026,10,stationary,0.271510,0.267655,0.275365
2050,10,stationary,0.271510,0.267655,0.275365
2026,10,nonstationary,0.268486,NaN,NaN
2050,10,nonstationary,0.266243,NaN,NaN
2026,25,stationary,0.293600,0.289745,0.297455
2050,25,stationary,0.293600,0.289745,0.297455
2026,25,nonstationary,0.290539,NaN,NaN
2050,25,nonstationary,0.288296,NaN,NaN
2026,50,stationary,0.307074,0.303219,0.310929


In [ ]:
_ = [gev.print_report(results[loc_ex], loc_ex) for loc_ex in results]

In [ ]:
mu0_samples = results[loc_id]['nonstationary']['mu0_samples']
mu1_samples = results[loc_id]['nonstationary']['mu1_samples']

In [ ]:
std(mu0_samples, ddof=1), std(mu1_samples, ddof=1)
#nonstationary['sigma_std'] = np.std(sigma_samples, ddof=1)
#nonstationary['xi_std'] = np.std(xi_samples, ddof=1)

## Compute For All (selected) Locations

In [ ]:
args_list = [(loc_id, dic_data_per_location[loc_id]) for loc_id in list(dic_data_per_location.keys())]

chunk_size = 500
chunks = [args_list[i:i + chunk_size] for i in range(0, len(args_list), chunk_size)]

results_all = {}
notes_dict = {}

for chunk_idx, chunk in enumerate(chunks, 1):
    print(f"Processing chunk {chunk_idx}/{len(chunks)} ({len(chunk)} locations)")
    
    with tqdm_joblib(tqdm(total=len(chunk), desc=f"Chunk {chunk_idx}")) as progress_bar:
        chunk_results = Parallel(n_jobs=-1)(
            delayed(gev.process_gev_per_location)(
                loc_id, df, trend_params='location', return_periods=return_periods,
                t_eval=ls_t_eval, uncertainty='fisher', B=150, seed=None, print_msg=False
            )
            for loc_id, df in chunk
        )
    
    for loc_id, res, notes in chunk_results:
        results_all[loc_id] = res
        notes_dict[loc_id] = notes

In [ ]:
_ = [gev.print_report(results_all[loc_ex], loc_ex) for loc_ex in results_all]

## Plot Figures 

NOTE the result overview refers to the bootstrapped (of Fisher information), hence the (occasional) difference between stationary and non-stationary approach. 

In [ ]:
print(f'Process siteID {loc_id}')

fig = dbplt.plot_pooled_analysis_v2(
    result=results[loc_id], site_id=loc_id, return_periods=return_periods,
    leg_comparison_x=0.075, leg_comparison_y=0.35, box_parameters_x=0.45,  box_parameters_y= 0.95,
    linestyle_trends = ['-', '--', '-.', ':', (0, (1, 1)), (0, (5, 10))], fontsize=12, figsize=(15, 7.5),
    display_results=True)

In [ ]:
dic_fig = dict(map(lambda loc_ex: (loc_ex, gev.plot_pooled_analysis_v2(
                    result=results_all[loc_ex], site_id=loc_ex, 
                    leg_comparison_x=0.075, leg_comparison_y=0.35, 
                    box_parameters_x=0.45,  box_parameters_y= 0.95,
                    linestyle_trends = ['-', '--', '-.', ':', (0, (1, 1)), (0, (5, 10))],
                    fontsize=12, figsize=(15, 7.5), display_results=True
                    )), list(results_all.keys())))

## Store Result(s)

In [ ]:
today_ = str(datetime.today().date().isoformat())   
path_child_folder = Path(path_export) / f"{today_}"

ut.save_pooled_results(results=results_all, data=None, base_dir=path_child_folder)

In [ ]:
fig_dir = Path(path_child_folder) / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

for loc_id, result in results_all.items():   
    fig = plot_pooled_analysis_v2(
        result=result, site_id=loc_id,
        leg_comparison_x=0.075, leg_comparison_y=0.35, 
        box_parameters_x=0.45,  box_parameters_y= 0.95,
        linestyle_trends = ['-', '--', '-.', ':', (0, (1, 1)), (0, (5, 10))],
        fontsize=12, figsize=(15, 7.5),
        )
    
    lat = str(result['LatLon'][0].round(3))
    lon = str(result['LatLon'][1].round(3))
    country = result['location_info'].split(' ')[-1].strip()

    fig.savefig(
        fig_dir / f"location_{loc_id}_{country}_{lat}_{lon}_pooledGEVanalysis.png", 
        dpi=150, bbox_inches="tight"
        )
    plt.close(fig)

# Annual Stationary GEV Analysis

For annual stationary GEV, each yearly fit likely has fewer data samples.
> For a small sample size for a 3-parameter GEV, especially with negative shape, Hessian becomes unstable because:
- Some years will have near-degenerate scale
- Some years will have shape near boundary
- Finite differences step outside parameter support
- Log-likelihood returns inf or nan
- Second derivative formula explodes


> Solution
Compute MLE for annual stationary for 3 parameters and then fit the regression incl uncertainty


In [ ]:
def fit_annual_gev_mle(df, ls_notes, col_data='storm_surge', col_year='sim_year'):
    """
    Fit stationary GEV MLEs for each year in the dataframe.
    Returns a dataframe with columns: ['year', 'mu', 'sigma', 'xi']
    """
    annual_max_per_year = df.groupby(col_year)[col_data].apply(list)
    records = []

    for year, data in annual_max_per_year.items():
        result, ls_notes = gev.mle_fitting(data, len(data), ls_notes)
        records.append((year, result))
    
    flattened = []
    for year, vals in records:
        row = {'year': year, **vals} 
        flattened.append(row)

    return DataFrame(flattened).sort_values('year').reset_index(drop=True)


def regress_location_trend(annual_df):
    """
    Fits linear trend: mu ~ standardized year
    Returns dict with mu0, mu1, SEs for both, plus mean and std of years
    """
    years = annual_df['year'].values
    years_mean = years.mean()
    years_std = years.std(ddof=0)  # same as np.std with population formula

    x_std = (years - years_mean) / years_std

    X = sm.add_constant(x_std)  # add intercept
    y = annual_df['location'].values
    model = sm.OLS(y, X).fit()

    return {
        'mu0': model.params[0],
        'mu1': model.params[1],
        'mu0_se': model.bse[0],
        'mu1_se': model.bse[1],
        'years_mean': years_mean,
        'years_std': years_std
    }
    

def fit_location(loc_id, df, ls_notes):
    """
    Fits annual stationary GEV MLEs and computes mu trend regression
    """
    annual_df = fit_annual_gev_mle(df, ls_notes=None)
    trend_results = regress_location_trend(annual_df)

    return loc_id, {
        'annual_mle': annual_df,
        'mu_trend': trend_results
    }


def fit_all_locations(dic_data_per_location, n_jobs=-1):
    results = Parallel(n_jobs=n_jobs, backend='loky')(
        delayed(fit_location)(loc_id, df, None) for loc_id, df in dic_data_per_location.items()
    )
    return dict(results)

In [ ]:
results_annual_stat_all = fit_all_locations(dic_data_per_location, n_jobs=-1)

In [ ]:
results_annual_stat_all[loc_id]['annual_mle']

In [ ]:
try:
    for loc_id, df_yearly in results_annual_stat_all.items():
        if loc_id in results:
            results[loc_id]['annual_stationary'] = df_yearly
        else:
            results[loc_id] = {'annual_stationary': df_yearly}

except:
    results = {}
    results[loc_id] = dict({'annual_stationary': results_annual_stat_all})

In [ ]:
def store_annual_stat_results(results_annual_stat, path_child_folder):
    os.makedirs(path_child_folder, exist_ok=True)

    filename = path_child_folder / 'stationary_per_year.pkl'
    with open(filename, "wb") as f:
        pickle.dump(results_annual_stat, f)


    print(f"Output stored in {filename}")

In [ ]:
try:
    dir_export = path_child_folder
except:
    today_ = str(datetime.today().date().isoformat())   
    dir_export = Path(path_export) / f"{today_}"
    
store_annual_stat_results(results_annual_stat_all, dir_export)

# Regression Analysis for Location Parameter μ

- Nonstationary pooled uses already weighting in likelihood > standard regression on fitted mu0/mu1
- Annual stationary needs to actively be included in regression when n_obs vary for years

In [ ]:
axes_color: str = '#333333'
markers_color: str = "#99E3DDFF"
colors_reg: list = ['#CAA5C2FF',  '#005C55FF']  

fs = 12

### Import Data if Needed

In [ ]:
dir_annual_stat = '../output/gev_analysis/2026-02-25/stationary_per_year.pkl'
dir_nonstat = '../output/gev_analysis/2026-02-24/nonstationary.pkl'

In [ ]:
with open(dir_annual_stat, "rb") as f:
    results_annual_stat_all = pickle.load(f)
    
    
with open(dir_nonstat, "rb") as f:
    results_nonstat_all = pickle.load(f)

In [ ]:
loc_id = list(dic_data_per_location.keys())[0]
results_per_location = results[loc_id]

## Prepare & Compute Regression

In [ ]:
factor_m_to_mm = 1000
z = 1.96

In [ ]:
years_mean = results_per_location['nonstationary']['years_mean']
years_std = results_per_location['nonstationary']['years_std']

In [ ]:
years_ = arange(hindcast_start, hindcast_end+1)
years_autoscaled = (years_ - years_mean) / years_std

In [ ]:
results_annual_stat_location = results_per_location['annual_stationary']

x_ans = results_annual_stat_location['annual_mle'].year.values
y_ans = results_annual_stat_location['annual_mle'].location.values*factor_m_to_mm
weights_ans = results_annual_stat_location['annual_mle'].n_obs.values

In [ ]:
results_reg_annual_stat = gev.annual_stationary_trend(results_annual_stat_location['annual_mle'])

intercept_ans = results_reg_annual_stat['mu_trend']['mu0']
slope_ans = results_reg_annual_stat['mu_trend']['mu1']

In [ ]:
mu0_ns = results_per_location['nonstationary']['params_hat'][0]*factor_m_to_mm
mu1_ns = results_per_location['nonstationary']['params_hat'][1]*factor_m_to_mm
mu0_std_ns = results_per_location['nonstationary']['mu0_std']*factor_m_to_mm
mu1_std_ns = results_per_location['nonstationary']['mu1_std']*factor_m_to_mm

mu_ns = mu0_ns + mu1_ns * years_autoscaled
mu_ns_ci_upper = mu_ns + z * sqrt(mu0_std_ns + (years_autoscaled**2) * mu1_std_ns)
mu_ns_ci_lower = mu_ns - z * sqrt(mu0_std_ns + (years_autoscaled**2) * mu1_std_ns)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

# annual-stationary GEV results
ax.scatter(
        x_ans, y_ans, s=weights_ans*2.5, 
        marker='o', color=markers_color, alpha=0.75, 
        label='fit result annual stationary GEV (size ~ n_obs)'
        )
for idx, row in results_annual_stat_location['annual_mle'].iterrows():
        ax.text(row.year, row.location*1005, f"{row.n_obs}", fontsize=8, alpha=0.6)

ax.plot(
        results_annual_stat_location['annual_mle']['year'], results_reg_annual_stat['mu_fit'], color=colors_reg[0], lw=1, 
        label=f'annual stationary lin.regression · y(t) = {slope_ans:.3f}·t + {intercept_ans:.3f}'
        )
ax.fill_between(
        results_annual_stat_location['annual_mle']['year'], results_reg_annual_stat['mu_ci_lower'], 
        results_reg_annual_stat['mu_ci_upper'], 
        color=colors_reg[0], alpha=0.3, lw=0, label='annual stationary – 95% CI'
        )

# non-stationary GEV
plt.plot(
        years_, mu_ns, color=colors_reg[1], ls='-.',
        label=f'non-stationary lin.regression · y(t) = {mu1_ns:.3f}·t + {mu0_ns:.3f}'
        )
ax.fill_between(years_, mu_ns_ci_lower, mu_ns_ci_upper, color=colors_reg[1], alpha=0.25, label='non-stationary – 95% CI')


# ----------------------------------------------------------------------------
# layout
leg = ax.legend(loc=0, edgecolor=axes_color, borderpad=.65, fontsize=fs*0.75)
leg.get_frame().set_linewidth(.5)

for spine in ax.spines.values():
        spine.set_visible(False)

ax.axhline(y=ax.get_ylim()[0], color=axes_color, linewidth=1.2, zorder=10)
ax.axvline(x=ax.get_xlim()[0], color=axes_color, linewidth=1.2, zorder=10)

ax.tick_params(axis='x', colors=axes_color)
ax.tick_params(axis='y', colors=axes_color)

ax.grid(True, alpha=0.3, color='lightgrey')
ax.set_title(
        f'Trend Analysis for Location Parameter μ for siteID {loc_id} – annual stationary vs non-stationary GEV', 
        loc='left', fontsize=fs*1.25
        )

ax.set_xlabel('Year', fontsize=fs)
ax.set_ylabel('GEV location parameter μ, mm', fontsize=fs)
plt.tight_layout()
